In [0]:
%sql
DESCRIBE TABLE observatorio_dev.bronze.embalses;

In [0]:
%sql
SELECT * FROM observatorio_dev.bronze.embalses LIMIT 10;

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    to_date,
    current_timestamp,
    max as spark_max,
    lit,
    count,
    when
)

from delta.tables import DeltaTable

In [0]:
bronze_table = "observatorio_dev.bronze.embalses"
silver_table = "observatorio_dev.silver.embalses"

print("Bronze table:", bronze_table)
print("Silver table:", silver_table)

In [0]:
last_ingestion_timestamp = (
    spark.table(silver_table)
    .select(
        spark_max("ingestion_timestamp").alias("last_ingestion_timestamp")
    )
    .collect()[0]["last_ingestion_timestamp"]
)

print("Último ingestion_timestamp cargado en Silver:", last_ingestion_timestamp)

In [0]:
bronze_df = spark.table(bronze_table)

if last_ingestion_timestamp is not None:
    bronze_df = bronze_df.filter(
        col("ingestion_timestamp") > lit(last_ingestion_timestamp)
    )

print("Registros nuevos desde Bronze:", bronze_df.count())

display(bronze_df.limit(20))

In [0]:
silver_df = (
    bronze_df
    .select(
        col("id"),
        trim(upper(col("codigo_embalse"))).alias("codigo_embalse"),
        trim(upper(col("nombre_embalse"))).alias("nombre_embalse"),
        col("latitud").cast("double").alias("latitud"),
        col("longitud").cast("double").alias("longitud"),

        col("source_file_name"),
        col("source_file_path"),
        col("ingestion_timestamp"),
        col("load_date"),

        current_timestamp().alias("silver_created_at"),
        current_timestamp().alias("silver_updated_at")
    )
    .dropDuplicates([
        "codigo_embalse",
    ])
)

print("Registros transformados para Silver:", silver_df.count())

display(silver_df.limit(20))

In [0]:
invalid_coordinates_df = (
    silver_df
    .filter(
        (col("latitud") < -90) |
        (col("latitud") > 90) |
        (col("longitud") < -180) |
        (col("longitud") > 180)
    )
)

print("Registros con coordenadas inválidas:", invalid_coordinates_df.count())

display(invalid_coordinates_df)

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("observatorio_dev.bronze.embalses")

reference_df = spark.table("observatorio_dev.reference.embalses_coordinates_manual")

silver_df = (
    bronze_df.alias("b")
    .join(
        reference_df.alias("r"),
        on="codigo_embalse",
        how="left"
    )
    .select(
        F.col("b.id"),
        F.col("b.codigo_embalse"),
        F.col("b.nombre_embalse"),

        F.coalesce(
            F.col("b.latitud").cast("double"),
            F.col("r.latitud_manual").cast("double")
        ).alias("latitud"),

        F.coalesce(
            F.col("b.longitud").cast("double"),
            F.col("r.longitud_manual").cast("double")
        ).alias("longitud"),

        F.when(
            F.col("b.latitud").isNotNull() & F.col("b.longitud").isNotNull(),
            F.lit("valid_raw")
        ).when(
            F.col("r.latitud_manual").isNotNull() & F.col("r.longitud_manual").isNotNull(),
            F.lit("enriched_reference")
        ).otherwise(
            F.lit("missing")
        ).alias("coordenadas_status"),

        F.when(
            F.col("b.latitud").isNotNull() & F.col("b.longitud").isNotNull(),
            F.lit("raw")
        ).when(
            F.col("r.latitud_manual").isNotNull() & F.col("r.longitud_manual").isNotNull(),
            F.col("r.coordenadas_source")
        ).otherwise(
            F.lit("not_available")
        ).alias("coordenadas_source"),

        F.when(
            F.col("b.latitud").isNull() | F.col("b.longitud").isNull(),
            F.coalesce(F.col("r.requires_manual_review"), F.lit(True))
        ).otherwise(
            F.lit(False)
        ).alias("requires_manual_review"),

        F.col("r.confidence"),
        F.col("r.source_name"),
        F.col("r.source_url"),
        F.col("r.coordinate_basis"),

        F.col("b.source_file_name"),
        F.col("b.source_file_path"),
        F.col("b.ingestion_timestamp"),
        F.col("b.load_date"),

        F.current_timestamp().alias("silver_created_at"),
        F.current_timestamp().alias("silver_updated_at")
    )
)

In [0]:
from delta.tables import DeltaTable

new_records_count = silver_df.count()

print("Registros nuevos a procesar:", new_records_count)

if new_records_count == 0:
    print("No hay registros nuevos para cargar en Silver.")

else:
    target = DeltaTable.forName(spark, silver_table)

    (
        target.alias("target")
        .merge(
            silver_df.alias("source"),
            """
            target.codigo_embalse = source.codigo_embalse
            """
        )
        .whenMatchedUpdate(set={
            "id": "source.id",
            "nombre_embalse": "source.nombre_embalse",
            "latitud": "source.latitud",
            "longitud": "source.longitud",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_updated_at": "source.silver_updated_at"
        })
        .whenNotMatchedInsert(values={
            "id": "source.id",
            "codigo_embalse": "source.codigo_embalse",
            "nombre_embalse": "source.nombre_embalse",
            "latitud": "source.latitud",
            "longitud": "source.longitud",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_created_at": "source.silver_created_at",
            "silver_updated_at": "source.silver_updated_at"
        })
        .execute()
    )

    print("MERGE ejecutado correctamente.")

In [0]:
%sql
SELECT * 
FROM observatorio_dev.silver.embalses
ORDER BY id